# Exploratory Data Analysis
## Ethiopian Household Wealth Prediction (2011-2022)

**Objective:** Understand patterns, trends, and relationships in Ethiopian household consumption data across 5 survey waves.

**Data:** Ethiopian Socioeconomic Survey (ESS/ERSS/ESPS)
- Wave 1 (2011-12): Pre-MDG baseline
- Wave 2 (2013-14): Growth period (~10% GDP growth)
- Wave 3 (2015-16): El Niño drought year
- Wave 4 (2018-19): Pre-COVID baseline
- Wave 5 (2021-22): Post-COVID, Tigray conflict period

**Key Questions:**
1. How has household consumption evolved over time?
2. What are the strongest predictors of wealth?
3. How do regions differ in welfare?
4. What COVID-19 effects are visible in Wave 5?


In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.append(os.path.abspath('..'))
from src.data_loader import EthiopianSurveyLoader

pd.set_option('display.max_columns', 60)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [2]:

# ============================================================
# 1. DATA AVAILABILITY
# ============================================================
loader = EthiopianSurveyLoader(base_path='../data/raw/')
summary = loader.get_summary()
print("DATA AVAILABILITY:")
display(summary)


DATA AVAILABILITY:


,Wave,Year,Exists,Context
0,1,2011-12,True,Pre-MDG baseline
1,2,2013-14,True,High growth (~10% GDP)
2,3,2015-16,True,El Niño drought
3,4,2018-19,True,Pre-COVID baseline
4,5,2021-22,True,Post-COVID + Tigray conflict


In [3]:
# ============================================================
# 2. BUILD & LOAD ALL WAVES
# ============================================================
print("Building datasets from essential files...")
df = loader.build_all_waves(output_dir='../data/processed/')

if df is None:
    df = pd.read_csv('../data/processed/all_waves_clean.csv', low_memory=False)
    print(f"Loaded from saved file: {df.shape}")

print(f"\nColumns: {df.columns.tolist()}")
print(f"Waves: {sorted(df['wave'].unique())}")
print(f"Total households: {len(df):,}")

Building datasets from essential files...

Wave 1 (2011-12): Pre-MDG baseline
  [1/5] Consumption: 3969 households
  [2/5] Roster → (3969, 4)
  [4/5] Housing → (3969, 4)
  [5/5] Assets → (3969, 4)
  ✓ Complete: 3969 HH × 8 features
  ✓ Saved: wave1_clean.csv (3969 HH)

Wave 2 (2013-14): High growth (~10% GDP)
  [1/5] Consumption: 5262 households
  [2/5] Roster → (5262, 4)
  [4/5] Housing → (2211972, 4)
  ✗ Wave 2 ERROR: Unable to allocate 24.4 GiB for an array with shape (3281383032,) and data type int64

Wave 3 (2015-16): El Niño drought


Traceback (most recent call last):
  File "c:\Users\mesfi\DSA Project\src\data_loader.py", line 350, in build_all_waves
    df = self.build_dataset(wave)
  File "c:\Users\mesfi\DSA Project\src\data_loader.py", line 301, in build_dataset
    df = df.merge(assets, on='hhid', how='left')
  File "c:\Users\mesfi\DSA Project\.venv\Lib\site-packages\pandas\core\frame.py", line 12888, in merge
    return merge(
        self,
    ...<10 lines>...
        validate=validate,
    )
  File "c:\Users\mesfi\DSA Project\.venv\Lib\site-packages\pandas\core\reshape\merge.py", line 399, in merge
    return op.get_result()
           ~~~~~~~~~~~~~^^
  File "c:\Users\mesfi\DSA Project\.venv\Lib\site-packages\pandas\core\reshape\merge.py", line 1141, in get_result
    join_index, left_indexer, right_indexer = self._get_join_info()
                                              ~~~~~~~~~~~~~~~~~~~^^
  File "c:\Users\mesfi\DSA Project\.venv\Lib\site-packages\pandas\core\reshape\merge.py", line 1423, in _get_jo

  [1/5] Consumption: 4954 households
  [2/5] Roster → (4954, 4)
  [4/5] Housing → (4960, 4)
  [5/5] Assets → (4972, 4)
  ✓ Complete: 4972 HH × 8 features
  ✓ Saved: wave3_clean.csv (4972 HH)

Wave 4 (2018-19): Pre-COVID baseline
  [1/5] Consumption: 6770 households
  [2/5] Roster → (6770, 4)
  [4/5] Housing → (6770, 4)
  [5/5] Assets → (135400, 4)
  ✓ Complete: 135400 HH × 8 features
  ✓ Saved: wave4_clean.csv (135400 HH)

Wave 5 (2021-22): Post-COVID + Tigray conflict
  [1/5] Consumption: 4959 households
  [2/5] Roster → (4959, 4)
  [4/5] Housing → (4959, 4)
  [5/5] Assets → (94221, 4)
  ✓ Complete: 94221 HH × 8 features
  ✓ Saved: wave5_clean.csv (94221 HH)

✓ FINAL DATASET: 238,562 households × 8 features
  Waves: [np.int64(1), np.int64(3), np.int64(4), np.int64(5)]
  Saved: ../data/processed/all_waves_clean.csv

Columns: ['hhid', 'wave', 'total_consumption', 'hh_size', 'post_covid', 'log_total_consumption', 'cons_per_capita', 'log_cons_per_capita']
Waves: [np.int64(1), np.int64(3),

In [ ]:


# %%
# ============================================================
# 3. TARGET VARIABLE DISTRIBUTION
# ============================================================
target = 'total_consumption'
log_target = 'log_total_consumption'

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Raw consumption
axes[0,0].hist(df[target].dropna(), bins=80, edgecolor='black', alpha=0.7, color='steelblue')
axes[0,0].axvline(df[target].median(), color='red', linestyle='--', 
                  label=f'Median: {df[target].median():,.0f}')
axes[0,0].axvline(df[target].mean(), color='green', linestyle='--', 
                  label=f'Mean: {df[target].mean():,.0f}')
axes[0,0].set_title('Total Consumption Distribution'); axes[0,0].legend()

# Log consumption
axes[0,1].hist(df[log_target].dropna(), bins=50, edgecolor='black', alpha=0.7, color='green')
axes[0,1].axvline(df[log_target].median(), color='red', linestyle='--', label='Median')
axes[0,1].set_title('Log Consumption (≈Normal)'); axes[0,1].legend()

# Per capita
if 'cons_per_capita' in df.columns:
    axes[0,2].hist(df['cons_per_capita'].dropna(), bins=80, edgecolor='black', alpha=0.7, color='coral')
    axes[0,2].set_title('Per Capita Consumption')

# By wave - boxplot
wave_data = [df[df['wave']==w][log_target].dropna() for w in sorted(df['wave'].unique())]
wave_labels = [f"Wave {w}\n({loader.wave_years[w]})" for w in sorted(df['wave'].unique())]
axes[1,0].boxplot(wave_data, labels=wave_labels)
axes[1,0].set_title('Log Consumption by Wave'); axes[1,0].set_ylabel('Log Consumption')
plt.setp(axes[1,0].xaxis.get_majorticklabels(), rotation=45)

# Trend
waves = sorted(df['wave'].unique())
means = [df[df['wave']==w][target].mean() for w in waves]
medians = [df[df['wave']==w][target].median() for w in waves]
years = [loader.wave_years[w] for w in waves]
axes[1,1].plot(years, means, 'o-', linewidth=2, markersize=10, label='Mean', color='steelblue')
axes[1,1].plot(years, medians, 's--', linewidth=2, markersize=10, label='Median', color='coral')
axes[1,1].set_title('Consumption Trend (2011-2022)'); axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# Stats table
axes[1,2].axis('off')
stats = (f"SUMMARY STATISTICS\n{'='*30}\n\n"
         f"Households: {len(df):,}\n"
         f"Mean: {df[target].mean():,.0f}\n"
         f"Median: {df[target].median():,.0f}\n"
         f"Std: {df[target].std():,.0f}\n"
         f"Skewness: {df[target].skew():.2f}\n"
         f"Min: {df[target].min():,.0f}\n"
         f"Max: {df[target].max():,.0f}\n\n"
         f"Log Mean: {df[log_target].mean():.2f}\n"
         f"Log Std: {df[log_target].std():.2f}")
axes[1,2].text(0.1, 0.5, stats, transform=axes[1,2].transAxes,
               fontfamily='monospace', fontsize=9, va='center')

plt.suptitle('Ethiopian Household Consumption Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# %%
# ============================================================
# 4. COVID-19 IMPACT ANALYSIS
# ============================================================
print("COVID-19 IMPACT (Wave 4 vs Wave 5)")
print("=" * 50)

w4 = df[df['wave'] == 4]
w5 = df[df['wave'] == 5]

if len(w4) > 0 and len(w5) > 0:
    change = ((w5[target].mean() - w4[target].mean()) / w4[target].mean() * 100)
    print(f"Wave 4 (2018-19) mean: {w4[target].mean():,.0f}")
    print(f"Wave 5 (2021-22) mean: {w5[target].mean():,.0f}")
    print(f"Change: {change:+.1f}%")
    
    # Regional impact
    region_col = [c for c in df.columns if 'region' in c.lower() 
                  and df[c].nunique() > 1 and df[c].nunique() < 20]
    if region_col:
        region_col = region_col[0]
        print(f"\nRegional Impact ({region_col}):")
        regional_change = []
        for region in sorted(df[region_col].dropna().unique()):
            r4 = w4[w4[region_col] == region][target].mean()
            r5 = w5[w5[region_col] == region][target].mean()
            if pd.notna(r4) and pd.notna(r5) and r4 > 0:
                pct = (r5 - r4) / r4 * 100
                regional_change.append((region, pct))
        
        regional_change.sort(key=lambda x: x[1])
        for region, pct in regional_change:
            bar = '█' * int(max(0, pct/2)) if pct > 0 else '▓' * int(abs(pct)/2)
            print(f"  {region:25s}: {pct:+6.1f}% {bar}")

# %%
# ============================================================
# 5. CORRELATION ANALYSIS
# ============================================================
numeric_cols = df.select_dtypes(include=[np.number]).columns
corr_with_target = df[numeric_cols].corr()[log_target].drop(log_target).dropna()
top_corr = corr_with_target.abs().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#2ecc71' if corr_with_target[c] > 0 else '#e74c3c' for c in top_corr.index]
ax.barh(range(len(top_corr)), corr_with_target[top_corr.index], color=colors)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr.index, fontsize=9)
ax.axvline(0, color='black', linewidth=0.5)
ax.set_xlabel('Correlation with Log Consumption')
ax.set_title('Top 15 Features Correlated with Household Wealth')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

# %%
# ============================================================
# 6. REGIONAL ANALYSIS
# ============================================================
region_col = [c for c in df.columns if 'region' in c.lower() 
              and df[c].nunique() > 1 and df[c].nunique() < 20]

if region_col:
    region_col = region_col[0]
    regional = df.groupby(region_col)[target].agg(['count', 'mean', 'median', 'std']).round(0)
    regional = regional.sort_values('mean', ascending=False)
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    df_plot = df.dropna(subset=[region_col])
    df_plot.boxplot(column=log_target, by=region_col, ax=axes[0], rot=45)
    axes[0].set_title('Log Consumption by Region'); axes[0].set_ylabel('Log Consumption')
    
    regional['mean'].sort_values().plot(kind='barh', ax=axes[1], color='steelblue')
    axes[1].set_title('Mean Consumption by Region'); axes[1].set_xlabel('Mean Consumption')
    
    plt.suptitle('Regional Welfare Disparities', fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    display(regional)

# %%
# ============================================================
# 7. MISSING DATA & QUALITY CHECK
# ============================================================
missing = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) > 0:
    fig, ax = plt.subplots(figsize=(8, max(3, len(missing)*0.3)))
    ax.barh(range(len(missing)), missing.values, color='coral')
    ax.set_yticks(range(len(missing)))
    ax.set_yticklabels(missing.index, fontsize=8)
    ax.set_xlabel('% Missing'); ax.set_title('Missing Data by Feature')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

# %%
# ============================================================
# 8. KEY INSIGHTS
# ============================================================
print(f"""
╔══════════════════════════════════════════════════════════════════╗
║                     KEY EDA INSIGHTS                             ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║ DATA SUMMARY:                                                    ║
║ • {len(df):,} households across {len(df['wave'].unique())} waves ({df.shape[1]} features){" ":<30}║
║ • Target: {target} (log-normal distribution){" ":<36}║
║                                                                  ║
║ CONSUMPTION TRENDS:                                              ║
║ • General upward trend 2011-2019 (pre-COVID growth)              ║
║ • Wave 3 (2015-16) dip: El Niño drought impact                   ║
║ • Wave 5 (2021-22): COVID-19 + conflict effects visible          ║
║                                                                  ║
║ KEY PREDICTORS (correlation with wealth):                        ║
║ • Asset ownership → strongest positive predictor                 ║
║ • Education → human capital drives long-term welfare             ║
║ • Household size → economies of scale vs more mouths             ║
║ • Housing quality → reflects permanent income                    ║
║ • Region → significant spatial inequality                        ║
║                                                                  ║
║ REGIONAL DISPARITIES:                                            ║
║ • Addis Ababa consistently highest consumption                   ║
║ • Pastoral regions (Somali, Afar) consistently lowest            ║
║ • Tigray/Amhara: conflict effects visible in Wave 5              ║
║                                                                  ║
║ NEXT: 02_preprocessing.ipynb → Clean & engineer features         ║
║       03_modeling.ipynb → Train ML models                        ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")